# Fine-Tuning Nemotron-Nano-4B on Trading Data
## A Complete Story: from Raw JSONL to a Deployed Trading-Specialist LLM

This notebook is a **deep walkthrough** -- not just code you run, but a complete explanation of
what is happening at every level: the transformer architecture, the memory layout, the gradient
flow, and every library involved.

By the end you will understand:
- Which specific matrices inside a 4B-parameter model we are changing
- How LoRA reduces 4B trainable parameters to ~40M
- How QLoRA compresses the frozen base from 16-bit to 4-bit
- Exactly how memory is allocated across model weights / optimizer / gradients / activations
- How the forward pass -> loss -> backward pass -> weight update cycle works
- What every library (Transformers, PEFT, TRL, bitsandbytes, MLX-LM) does at each step

**Prerequisites:** Basic Python. No advanced maths required -- every formula is explained in words.

---
## Chapter 1 -- The Big Picture: What Are We Actually Doing?

### The problem

We have a 4-billion-parameter language model (`nvidia/Nemotron-Nano-4B-Instruct`) that knows
everything about the internet -- but nothing about:
- GNPA (Gross Non-Performing Assets) of Indian banks
- NSE XBRL quarterly filings
- Demand/supply zone trading
- What a PCR of 0.87 means for Canara Bank's option chain

We want it to speak fluent "Indian equity trader" -- producing structured JSON decisions
like `{"action":"BUY","stop_loss":116.5,"reasoning":"EMA stack aligns with demand zone..."}`.

### The solution: supervised fine-tuning (SFT)

We show the model 500+ examples of:
1. **Input:** market data (candles, EMA, zones, fundamentals, trade history)
2. **Expected output:** the correct JSON decision + reasoning

The model learns to map inputs -> outputs by adjusting its internal weights.

### Why not just use the base model with better prompts?

You could -- but:
- The base model wastes tokens re-explaining what GNPA is every call
- It frequently hallucinates field names in JSON
- It doesn't know your specific schema
- After fine-tuning, the model **is** the trading specialist -- no prompt engineering needed

In [ ]:
# What does a training example look like?
import json

example = {
    "messages": [
        {
            "role": "system",
            "content": "You are a disciplined NSE equity trader. Protect capital first. Output JSON only."
        },
        {
            "role": "user",
            "content": """[CANBK 2024-10-10]
C:119.78 (+1.9%) | 5d:116.8-123.5 C@44pct
EMA: 50@118.9 < 100@120.1 < 200@122.5 [BEARISH]
VWAP:119.2 C+0.49%
S:116.5-117.2(-2.5%)  R:122.0-123.5(+2.5%)
Fund(2024-Q2): PAT Rs.5381cr | P/E 9.2
HISTORY:
01  BUY  SL_HIT   -52   9948  wait for EMA alignment
CAP:9948 (death<2000)"""
        },
        {
            "role": "assistant",
            "content": json.dumps({
                "action": "HOLD",
                "quantity": 0,
                "stop_loss": 0,
                "take_profit": 0,
                "risk_reward": 0,
                "reasoning": "EMA50 < EMA100 < EMA200 -- full bearish stack. Episode 1 lesson: wait for EMA alignment before entering. No valid setup.",
                "confidence": 0.75,
                "key_risk": "Missing break above EMA100 for confirmation"
            }, indent=2)
        }
    ]
}

# This is one JSONL line in data/trading_train.jsonl
print("System prompt length:", len(example["messages"][0]["content"]), "chars")
print("User prompt length:", len(example["messages"][1]["content"]), "chars")
print("Assistant output length:", len(example["messages"][2]["content"]), "chars")
print()
print("Assistant output:")
print(example["messages"][2]["content"])

---
## Chapter 2 -- Inside the Transformer: What We Are Modifying

### The architecture of Nemotron-Nano-4B

Nemotron-Nano-4B is built on the **Mistral architecture** -- a stack of 32 identical
transformer blocks. Each block has:

```
Transformer Block (x32)
+-- RMSNorm                         <- normalise the input
+-- Multi-Head Self-Attention
|   +-- Q projection  (q_proj)      <- "What am I looking for?"
|   +-- K projection  (k_proj)      <- "What do I have to offer?"
|   +-- V projection  (v_proj)      <- "What is my actual content?"
|   +-- O projection  (o_proj)      <- "Combine the attended values"
+-- RMSNorm
+-- Feed-Forward Network (MLP)
    +-- gate_proj                   <- gating mechanism
    +-- up_proj                     <- expand to hidden dim
    +-- down_proj                   <- compress back
```

**LoRA adds adapter matrices to all 7 projection layers** (q,k,v,o,gate,up,down).
The rest of the model (norms, embeddings, LM head) stays completely frozen.

### Why these 7 layers specifically?

- **Q, K, V projections:** control what the model pays attention to.
  Fine-tuning these teaches the model to attend to "GNPA", "EMA stack", "S/R zone"
  in a trading context rather than general language patterns.
- **O projection:** combines the attended values -- controls what information flows forward.
- **gate, up, down projections:** the feed-forward network processes the attended features.
  This is where factual knowledge is stored -- fine-tuning here teaches trading facts.

### Dimensions in Nemotron-Nano-4B

```
Hidden dimension (d_model):  4096
Number of heads:             32
Head dimension (d_head):     128  (= 4096/32)
FFN intermediate:            14336
Number of layers:            32
Vocabulary size:             32000
```

In [ ]:
# Visualise the attention mechanism for a trading-specific input
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Simulated attention weights for "EMA50 < EMA100 < EMA200 [BEARISH]"
# This shows what a fine-tuned vs base model attends to
tokens = ["EMA50", "<", "EMA100", "<", "EMA200", "[BEARISH]", "->", "HOLD"]
n = len(tokens)

# Base model: roughly uniform attention (doesn't know trading)
base_attn = np.random.dirichlet(alpha=[1]*n, size=n)

# Fine-tuned: concentrates on the key trading signals
ft_attn = np.zeros((n, n))
# HOLD decision attends to BEARISH tag and EMA relationships
ft_attn[7] = [0.18, 0.02, 0.15, 0.02, 0.15, 0.40, 0.04, 0.04]  # HOLD attends to BEARISH + EMAs
for i in range(7):
    ft_attn[i] = np.random.dirichlet(alpha=[2 if j==i else 0.5 for j in range(n)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#0f1117")
for ax, attn, title in zip(axes, [base_attn, ft_attn],
                            ["Base model (untrained)\nDiffuse attention -- no trading knowledge",
                             "Fine-tuned model\nConcentrated on trading signals"]):
    im = ax.imshow(attn, cmap="YlOrRd", aspect="auto", vmin=0, vmax=0.5)
    ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha="right",
                                                  fontsize=10, color="white")
    ax.set_yticks(range(n)); ax.set_yticklabels(tokens, fontsize=10, color="white")
    ax.set_title(title, color="white", fontsize=11, pad=10)
    ax.set_facecolor("#131722")
    plt.colorbar(im, ax=ax)

fig.suptitle("Attention weights -- what the model focuses on when deciding HOLD",
             color="white", fontsize=12)
fig.patch.set_facecolor("#0f1117")
plt.tight_layout()
plt.savefig("attention_viz.png", dpi=120, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Fine-tuned model correctly concentrates attention on [BEARISH] and EMA ordering.")

---
## Chapter 3 -- Memory Math: Where Does All the VRAM Go?

Training an LLM requires 4 distinct memory buckets. Understanding these explains
**why QLoRA is necessary** on consumer hardware.

### The 4 memory buckets

```
1. MODEL WEIGHTS      the frozen base model parameters
2. ADAPTER WEIGHTS    the trainable LoRA A and B matrices
3. OPTIMIZER STATES   Adam keeps running mean + variance per trainable param
4. ACTIVATIONS        intermediate values stored during forward pass for backprop
```

### Full fine-tune vs LoRA vs QLoRA

```
Full fine-tune (float32):
  Model weights:     4B x 4 bytes = 16 GB
  Optimizer (Adam):  4B x 8 bytes = 32 GB  (mean + variance)
  Gradients:         4B x 4 bytes = 16 GB
  Activations:       ~6 GB  (depends on batch size + seq length)
  -----------------------------------------
  TOTAL:             ~70 GB  <- needs A100 80GB

LoRA (bfloat16 base):
  Model weights:     4B x 2 bytes = 8 GB   (frozen, bfloat16)
  Adapter weights:   40M x 2 bytes = 80 MB
  Optimizer (Adam):  40M x 8 bytes = 320 MB  (only adapters!)
  Gradients:         40M x 4 bytes = 160 MB
  Activations:       ~4 GB
  -----------------------------------------
  TOTAL:             ~13 GB  <- needs RTX 3090 / A10G

QLoRA (NF4 base + bfloat16 adapters):
  Model weights:     4B x 0.5 bytes = 2 GB  (4-bit NF4)
  Adapter weights:   40M x 2 bytes = 80 MB
  Optimizer (Adam):  40M x 8 bytes = 320 MB
  Gradients:         40M x 4 bytes = 160 MB
  Activations:       ~3 GB  (gradient checkpointing helps)
  -----------------------------------------
  TOTAL:             ~5.5 GB  <- fits on free Colab T4 / M5 Air!
```

### Why does Adam need 2x the parameters?

Adam (Adaptive Moment Estimation) maintains two running averages per parameter:
- **m** (first moment): exponential moving average of the gradients
- **v** (second moment): exponential moving average of the squared gradients

Update rule:
```
m = beta? x m + (1-beta?) x gradient           (momentum)
v = beta? x v + (1-beta?) x gradient2          (adaptive scaling)
weight = weight - lr x m / (sqrtv + eps)       (parameter update)
```

With LoRA we only do this for the 40M adapter parameters, not 4B.
That's why optimizer memory drops from 32 GB to 320 MB -- a **100x reduction**.

In [ ]:
# Memory budget calculator
def memory_budget(method: str, batch_size: int = 2, seq_len: int = 1024) -> dict:
    """
    Calculate VRAM requirements for each training method.
    
    Precision sizes:
      float32  = 4 bytes per param
      bfloat16 = 2 bytes per param  
      int8     = 1 byte per param
      NF4      = 0.5 bytes per param (4-bit)
    """
    B = 1e9  # billion
    M = 1e6  # million
    GB = 1024**3
    
    total_params = 4 * B        # Nemotron-Nano-4B
    lora_params  = 40 * M       # ~40M trainable LoRA params
    
    budgets = {
        "Full fine-tune (fp32)": {
            "model_weights":    total_params * 4,           # fp32
            "optimizer":        total_params * 8,           # Adam: 2 x fp32
            "gradients":        total_params * 4,           # fp32
            "activations":      batch_size * seq_len * 4096 * 4 * 4  # rough estimate
        },
        "LoRA (bf16 base)": {
            "model_weights":    total_params * 2,           # bfloat16, frozen
            "adapter_weights":  lora_params * 2,            # bfloat16
            "optimizer":        lora_params * 8,            # Adam on adapters only
            "gradients":        lora_params * 4,            # fp32 grads
            "activations":      batch_size * seq_len * 4096 * 4 * 2  # with grad ckpt
        },
        "QLoRA 8-bit": {
            "model_weights":    total_params * 1,           # int8
            "adapter_weights":  lora_params * 2,
            "optimizer":        lora_params * 8,
            "gradients":        lora_params * 4,
            "activations":      batch_size * seq_len * 4096 * 4 * 2
        },
        "QLoRA 4-bit NF4": {
            "model_weights":    total_params * 0.5,         # NF4
            "adapter_weights":  lora_params * 2,
            "optimizer":        lora_params * 8,
            "gradients":        lora_params * 4,
            "activations":      batch_size * seq_len * 4096 * 4 * 1  # grad ckpt
        },
    }
    
    print(f"\nMemory budget (batch={batch_size}, seq_len={seq_len}):")
    print(f"{'Method':<25} {'Weights':>8} {'Optim':>8} {'Grads':>8} {'Activ':>8} {'TOTAL':>9}")
    print("-" * 70)
    for method_name, b in budgets.items():
        w = b.get("model_weights",0) + b.get("adapter_weights",0)
        o = b.get("optimizer",0)
        g = b.get("gradients",0)
        a = b.get("activations",0)
        total = w + o + g + a
        fits = "M5 Air?" if total < 12*GB else ("T4?" if total < 16*GB else ("A100?" if total < 80*GB else "A100 80G only"))
        print(f"  {method_name:<23} {w/GB:>7.1f}G {o/GB:>7.1f}G {g/GB:>7.1f}G {a/GB:>7.1f}G  {total/GB:>7.1f}G  {fits}")

memory_budget("all", batch_size=2, seq_len=1024)

---
## Chapter 4 -- LoRA: The Core Idea

**Library:** `peft` (Parameter-Efficient Fine-Tuning) by HuggingFace

### The mathematical insight

Every weight matrix in a transformer is a 2D matrix. When we fine-tune on a specific task,
the **change** in that matrix (?W) has a much lower rank than the matrix itself.

This is the LoRA hypothesis: **task-specific adaptations are low-rank**.

Instead of learning ?W directly (which would require storing another dxd matrix),
we decompose it as two small matrices:

```
?W = B x A

where:
  A ? R^(r x d_in)    random Gaussian init, r << d
  B ? R^(d_out x r)   zero init (so ?W = 0 at start)

During forward pass:
  output = Wxx + (BxA)xx x (alpha/r)
  
  alpha/r is a scaling factor (alpha=32, r=16 -> scale=2.0)
```

**Why does this work?**

Neural networks don't need every dimension to represent new knowledge.
The actual directions in weight space that matter for trading concepts
(like "if EMA50 < EMA200 -> bearish signal") require far fewer than 4096 dimensions.
r=16 gives 16 orthogonal directions -- enough to encode the core trading vocabulary.

### What PEFT does under the hood

```python
# This is what get_peft_model() does internally for each linear layer:
class LoraLinear(nn.Module):
    def __init__(self, original: nn.Linear, r: int, alpha: float):
        self.original = original    # frozen W (4096 x 4096, bfloat16)
        self.lora_A   = nn.Linear(original.in_features,  r, bias=False)  # A matrix
        self.lora_B   = nn.Linear(r, original.out_features, bias=False)  # B matrix
        self.scaling  = alpha / r
        
        # Init: A = random Gaussian, B = zeros
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        
        # Freeze the original
        for p in self.original.parameters():
            p.requires_grad = False
    
    def forward(self, x):
        return self.original(x) + self.lora_B(self.lora_A(x)) * self.scaling
```

PEFT wraps every target layer with this `LoraLinear` and marks only A and B as `requires_grad=True`.
The optimizer then only sees and updates those ~40M parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# -- Visualise LoRA decomposition -----------------------------------------------
fig = plt.figure(figsize=(16, 6), facecolor="#0f1117")
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

d, r = 64, 4  # scaled down for visualisation (real: d=4096, r=16)
np.random.seed(42)

# Simulate: A task-specific weight matrix has low intrinsic rank
# (in reality, fine-tuning changes are empirically low-rank)
true_delta = np.outer(np.random.randn(d), np.random.randn(d))
# Low-rank approximation via SVD
U, S, Vt = np.linalg.svd(true_delta)
rank_r_approx = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]

# Panel 1: Original weight matrix W
ax1 = fig.add_subplot(gs[0])
im1 = ax1.imshow(np.random.randn(d, d)*0.1, cmap="RdBu", vmin=-0.5, vmax=0.5)
ax1.set_title(f"Original W\n({d}x{d} = {d*d:,} params)\n[FROZEN in bfloat16]",
              color="white", fontsize=10)
ax1.axis("off")

# Panel 2: LoRA decomposition
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor("#131722")
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis("off")

# Draw A matrix
ax2.add_patch(plt.Rectangle((0.5, 4), r*0.4, d*0.08, color="#26a69a", alpha=0.8))
ax2.text(0.5 + r*0.2, 4 + d*0.04, f"A\n({r}x{d})", ha="center", va="center",
         color="white", fontsize=11, fontweight="bold")
ax2.text(0.5 + r*0.2, 3.5, f"{r*d:,} params", ha="center", color="#94a3b8", fontsize=9)

ax2.text(5, 7, "x", ha="center", va="center", color="white", fontsize=20, fontweight="bold")

# Draw B matrix
ax2.add_patch(plt.Rectangle((6, 3), d*0.04, r*0.4, color="#6d8dff", alpha=0.8))
ax2.text(6 + d*0.02, 3 + r*0.2, f"B\n({d}x{r})", ha="center", va="center",
         color="white", fontsize=11, fontweight="bold")
ax2.text(6 + d*0.02, 2.5, f"{d*r:,} params", ha="center", color="#94a3b8", fontsize=9)

ax2.text(5, 1.5, f"?W = B x A = {r*d + d*r:,} params\nvs {d*d:,} in full W\n({100*(r*d+d*r)/(d*d):.1f}% overhead)",
         ha="center", va="center", color="#fbbf24", fontsize=10)
ax2.set_title("LoRA decomposition\n(rank r=4, scaled view)", color="white", fontsize=10)

# Panel 3: Approximation quality
ax3 = fig.add_subplot(gs[2])
errors = [np.linalg.norm(true_delta - U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]) / np.linalg.norm(true_delta)
          for k in range(1, d//2)]
ax3.plot(range(1, d//2), errors, color="#26a69a", lw=2)
ax3.axvline(r, color="#ef5350", ls="--", lw=1.5, label=f"r={r} (our setting)")
ax3.axhline(errors[r-1], color="#f59e0b", ls=":", lw=1)
ax3.fill_between(range(1, r+1), 0, errors[:r], alpha=0.2, color="#26a69a")
ax3.set_xlabel("Rank r", color="white"); ax3.set_ylabel("Reconstruction error", color="white")
ax3.set_title(f"Approximation quality\nr={r} captures most signal", color="white", fontsize=10)
ax3.set_facecolor("#131722"); ax3.tick_params(colors="white")
ax3.legend(facecolor="#1e2230", labelcolor="white")
for sp in ax3.spines.values(): sp.set_edgecolor("#2a2e39")
ax3.text(r+0.5, errors[r-1]*1.05, f"{(1-errors[r-1])*100:.0f}% captured", color="#f59e0b", fontsize=9)

for ax in [ax1, ax3]: ax.set_facecolor("#131722")
fig.suptitle("LoRA: decompose the weight update ?W into two small matrices A and B",
             color="white", fontsize=13)
plt.savefig("lora_decomposition.png", dpi=120, bbox_inches="tight", facecolor="#0f1117")
plt.show()

---
## Chapter 5 -- QLoRA: Quantising the Frozen Base

**Library:** `bitsandbytes` by Tim Dettmers

### The problem LoRA alone doesn't solve

Even with LoRA, we still need to load the full 4B model in memory
(frozen, but present) to do the forward pass. At bfloat16, that's 8 GB.
On a T4 (16 GB) or M5 Air (16 GB), adding optimizer states pushes us over.

### The QLoRA solution: NF4 quantisation

Instead of storing each weight as a 16-bit float, store it as a **4-bit integer**.

```
bfloat16:  0 10000000 0000000000000000  (16 bits: 1 sign + 7 exponent + 8 mantissa)
NF4:       0101                         (4 bits: index into a 16-value lookup table)
```

**Normal Float 4 (NF4)** is specially designed for neural network weights:
- Neural network weights follow a roughly normal (Gaussian) distribution
- NF4's 16 quantisation levels are spaced to minimise error on normal distributions
- This makes NF4 better than uniform 4-bit for LLMs

### Double quantisation

bitsandbytes also quantises the *quantisation constants* themselves:
```
Each block of 64 weights has a scaling constant (float32 = 4 bytes)
Double-quant compresses that constant to 8 bits (~0.5 bytes)
Savings: 4 bytes -> 0.5 bytes per 64 params = 0.056 bytes/param extra saved
```

### Forward pass with QLoRA

```
Storage (NF4):    #### (4 bits per weight)     <- on disk / in GPU memory
                        ?  dequantise on-the-fly
Compute (bf16):   ################ (16 bits)   <- during matrix multiply
                        ?  gradient flows here
LoRA adapters:    ################ (16 bits)   <- these accumulate updates
```

The base weights are **never** updated. Gradients pass through the dequantised values,
but only LoRA A and B matrices accumulate gradient updates.

In [ ]:
# Demonstrate NF4 quantisation
import numpy as np
import matplotlib.pyplot as plt

# NF4 quantisation levels (actual values from bitsandbytes)
NF4_LEVELS = np.array([
    -1.0000, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911,  0.0000,
     0.0796,  0.1609,  0.2461,  0.3379,  0.4407,  0.5626,  0.7229,  1.0000
])

def nf4_quantise(weights: np.ndarray) -> tuple[np.ndarray, float]:
    """Quantise weights to NF4.  Returns (indices, scale_factor)."""
    max_abs = np.max(np.abs(weights)) + 1e-8
    normalised = weights / max_abs           # scale to [-1, 1]
    # Find nearest NF4 level for each weight
    indices = np.argmin(np.abs(normalised[:, None] - NF4_LEVELS[None, :]), axis=1)
    return indices.astype(np.uint8), max_abs

def nf4_dequantise(indices: np.ndarray, scale: float) -> np.ndarray:
    """Restore approximate weights from NF4 indices."""
    return NF4_LEVELS[indices] * scale

# Simulate a small weight block (64 weights from q_proj)
np.random.seed(42)
original = np.random.randn(64) * 0.02   # typical LLM weight scale

indices, scale = nf4_quantise(original)
restored = nf4_dequantise(indices, scale)
error = original - restored

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor="#0f1117")

ax = axes[0]
ax.hist(original, bins=20, color="#6d8dff", alpha=0.8, density=True)
x = np.linspace(original.min(), original.max(), 100)
ax.plot(x, np.exp(-x**2/(2*0.02**2))/(0.02*np.sqrt(2*np.pi)), 
        color="#fbbf24", lw=2, label="Gaussian fit")
ax.vlines(NF4_LEVELS * scale, 0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 50,
          colors="#ef5350", alpha=0.4, lw=0.8, label="NF4 levels")
ax.set_title("Original weights\n(Gaussian distribution)", color="white")
ax.set_facecolor("#131722"); ax.tick_params(colors="white")
ax.legend(facecolor="#1e2230", labelcolor="white", fontsize=8)

ax = axes[1]
ax.bar(range(16), np.bincount(indices, minlength=16), color="#26a69a", alpha=0.8)
ax.set_title("NF4 level usage\n(non-uniform spacing captures Gaussian better)", color="white")
ax.set_xlabel("NF4 level index (0-15)", color="white")
ax.set_facecolor("#131722"); ax.tick_params(colors="white")

ax = axes[2]
ax.scatter(original, error, alpha=0.6, color="#f59e0b", s=20)
ax.axhline(0, color="white", lw=0.8, ls="--")
rmse = np.sqrt(np.mean(error**2))
max_err = np.max(np.abs(error))
ax.set_title(f"Quantisation error\nRMSE={rmse:.6f}, Max={max_err:.6f}", color="white")
ax.set_xlabel("Original weight", color="white")
ax.set_ylabel("Error (original - restored)", color="white")
ax.set_facecolor("#131722"); ax.tick_params(colors="white")

bits_original = 64 * 32  # float32 bits
bits_nf4 = 64 * 4 + 32   # 4-bit indices + one float32 scale constant
print(f"Memory: float32 = {bits_original//8} bytes | NF4 = {bits_nf4//8} bytes | "
      f"Compression ratio: {bits_original/bits_nf4:.1f}x")
print(f"Quality: RMSE = {rmse:.6f} | Max error = {max_err:.6f} | "
      f"Relative error = {rmse/np.std(original)*100:.2f}%")

for ax in axes: 
    for sp in ax.spines.values(): sp.set_edgecolor("#2a2e39")
fig.suptitle("NF4 Quantisation: store 4-bit integers, dequantise to bf16 for compute",
             color="white", fontsize=12)
plt.tight_layout()
plt.savefig("nf4_quantisation.png", dpi=120, bbox_inches="tight", facecolor="#0f1117")
plt.show()

---
## Chapter 6 -- Data Loading and Tokenisation

**Libraries:** `datasets` (HuggingFace), `transformers` AutoTokenizer

### From JSONL to tokens

The model doesn't see text -- it sees integers. The tokeniser converts text to token IDs
using a vocabulary of 32,000 sub-word units (Byte-Pair Encoding for Nemotron).

```
"EMA50 < EMA100 < EMA200 [BEARISH]"
        ?  tokenise
[  EMA,  50,  ?<,  ?EMA, 100,  ?<,  ?EMA, 200,  ?[, BE, AR, ISH,  ]]
[11120, 2135,  529, 27100, 2753,  529, 27100, 8456, 518, 14750, 6246, 24120, 60]
```

### The chat template

Nemotron uses the ChatML format:
```
<|im_start|>system
You are a disciplined NSE equity trader...<|im_end|>
<|im_start|>user
[CANBK 2024-10-10]...<|im_end|>
<|im_start|>assistant
{"action": "HOLD",...}<|im_end|>
```

Special tokens: `<|im_start|>` = token 32002, `<|im_end|>` = token 32003.

### The critical training trick: only compute loss on assistant tokens

```
Input tokens:   [system...] [user...]   [assistant...]
Loss mask:      [   0    ]  [  0  ]     [     1      ]
                   ? ignored               ? only these tokens generate loss
```

`DataCollatorForCompletionOnlyLM` (from TRL) sets loss = 0 for all system and user tokens.
The model is **only penalised** for getting the assistant's JSON wrong -- not for failing
to repeat the market data back.

This is the key to efficient fine-tuning: teaching the model to decide, not to memorise inputs.

In [ ]:
# Demonstrate tokenisation and the loss mask
print("=" * 60)
print("Chapter 6: Tokenisation walkthrough")
print("=" * 60)

# -- Simulate chat template application --------------------------------------
conversation = [
    {"role": "system",    "content": "You are a disciplined NSE equity trader."},
    {"role": "user",      "content": "[CANBK 2024-10-10]\nC:119.78 EMA:[BEARISH]"},
    {"role": "assistant", "content": '{"action":"HOLD","reasoning":"bearish EMA stack"}'},
]

# ChatML format applied manually (what apply_chat_template does)
def apply_chatml(messages):
    parts = []
    for m in messages:
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n")
    return "".join(parts)

full_text = apply_chatml(conversation)
print("Full conversation after chat template:")
print(full_text)

# -- Demonstrate the loss mask -------------------------------------------------
print("\nLoss mask (simplified -- which tokens contribute to training loss):")
print("-" * 60)
for i, m in enumerate(conversation):
    role = m["role"]
    text = m["content"][:40] + ("..." if len(m["content"]) > 40 else "")
    # Only assistant tokens have loss = 1
    loss = "? TRAIN" if role == "assistant" else "? ignore"
    grad = "<-  gradients flow here" if role == "assistant" else "   (no gradient)"
    print(f"  [{role:<10}] {loss}  |  '{text}'  {grad}")

print()
print("The model learns WHAT TO DECIDE -- not to repeat the market data.")
print("This is why compact prompts (188 tokens) train faster than verbose ones (990 tokens).")
print("Fewer ignored tokens = more efficient gradient signal per batch.")

# -- Token count breakdown ---------------------------------------------------
import json

training_example = {
    "messages": [
        {"role":"system","content":"You are a disciplined NSE equity trader. Output JSON only."},
        {"role":"user","content":"[CANBK 2024-10-10]\nC:119.78 (+1.9%) | 5d:116.8-123.5 C@44pct\nEMA: 50@118.9 < 100@120.1 < 200@122.5 [BEARISH]\nVWAP:119.2 C+0.49%\nS:116.5-117.2(-2.5%)  R:122.0-123.5(+2.5%)\nFund(2024-Q2): PAT Rs.5381cr | P/E 9.2\nHISTORY:\n01  BUY  SL_HIT   -52   9948  wait for EMA alignment\nCAP:9948"},
        {"role":"assistant","content":json.dumps({"action":"HOLD","quantity":0,"stop_loss":0,"take_profit":0,"risk_reward":0,"reasoning":"EMA50<EMA100<EMA200 bearish. Ep1 lesson: wait for alignment.","confidence":0.75,"key_risk":"no confirmation"})}
    ]
}

# Rough token count (~= 4 chars per token)
for m in training_example["messages"]:
    n_chars = len(m["content"])
    n_tokens = n_chars // 4
    trained = "YES <- gradient" if m["role"] == "assistant" else "no"
    print(f"  {m['role']:<12}: ~{n_tokens:3d} tokens | trained on: {trained}")

total = sum(len(m["content"]) for m in training_example["messages"]) // 4
assistant_tokens = len(training_example["messages"][2]["content"]) // 4
print(f"\n  Total: ~{total} tokens | Trained on: ~{assistant_tokens} ({assistant_tokens/total*100:.0f}%)")
print(f"  \n  With compact_prompt.py: {188} tokens total -> {60} assistant -> better signal density")

---
## Chapter 7 -- The Training Loop: Forward -> Loss -> Backward -> Update

**Library:** `trl` SFTTrainer (wraps HuggingFace Trainer)

This is the heart of the whole process. Every training step has 4 phases:

### Phase 1: Forward pass

The batch of tokenised text flows through all 32 transformer layers.
At each layer, every `LoraLinear` computes:
```
output = W x input + (B x A x input) x (alpha/r)
              ?                ?
         original          tiny adapter
         (frozen)           (trainable)
```
The model produces a logit (probability) for every token in the vocabulary.

### Phase 2: Loss computation

For each **assistant token** in the batch, we compute the cross-entropy loss:
```
loss = -log P(correct_token | context)
```
If the model predicted `{"action":"BUY"...}` but the label is `{"action":"HOLD"...}`,
the loss for the "action" tokens is high. If it predicted correctly, loss is near 0.

The mean loss over the batch and sequence is the scalar we optimise.

### Phase 3: Backward pass (backpropagation)

PyTorch's autograd computes `?loss/?param` for every parameter with `requires_grad=True`.
Since only LoRA A and B have `requires_grad=True`, gradients only flow to those 40M params.

However, gradients still **pass through** the frozen base weights during backprop --
they just don't accumulate into those weights. This is what `prepare_model_for_kbit_training()`
sets up: the frozen base can still participate in the gradient computation chain.

### Phase 4: Optimizer step (Adam 8-bit)

```
For each trainable parameter ? (in LoRA A and B matrices):
  g  = ?loss/??                         # gradient from backward pass
  m  = beta? x m  + (1-beta?) x g            # update running mean    (beta?=0.9)
  v  = beta? x v  + (1-beta?) x g2           # update running variance (beta?=0.999)
  ?  = ? - lr x m? / (sqrtv? + eps)           # update parameter
```

`adamw_bnb_8bit` (from bitsandbytes) stores m and v in 8-bit -- halving optimizer memory
from 320 MB to 160 MB, with negligible accuracy loss.

### Gradient accumulation

With `gradient_accumulation_steps=8` and `batch_size=2`:
- Run forward+backward 8 times (each with 2 examples)
- Accumulate (add) gradients without zeroing
- After 8 micro-batches: do **one** optimizer step
- Effective batch = 2 x 8 = 16 examples

This mimics a batch size of 16 using only 2x-worth of activations in memory.

In [ ]:
# Visualise one complete training step
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 4, figsize=(16, 5), facecolor="#0f1117")
step_titles = [
    "1. Forward Pass\n(compute predictions)",
    "2. Loss\n(cross-entropy on\nassistant tokens only)",
    "3. Backward Pass\n(gradients for A and B)",
    "4. Optimizer Step\n(Adam updates A and B)"
]
step_colors = ["#6d8dff", "#ef5350", "#f59e0b", "#26a69a"]

for i, (ax, title, col) in enumerate(zip(axes, step_titles, step_colors)):
    ax.set_facecolor("#131722")
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
    ax.set_title(title, color="white", fontsize=10, pad=10)

    if i == 0:  # Forward pass
        for j, (y, label, c) in enumerate([
            (8.5, "Input tokens", "#94a3b8"),
            (7.2, "Transformer layer 1", "#6d8dff"),
            (6.0, "   Wxx + BAx (LoRA)", "#a78bfa"),
            (4.8, "Transformer layer 2..32", "#6d8dff"),
            (3.5, "LM Head -> logits", "#6d8dff"),
            (2.2, "Output probabilities", "#94a3b8"),
        ]):
            ax.text(5, y, label, ha="center", va="center", color=c, fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="#1e2230", 
                             edgecolor=col, alpha=0.8) if "layer" in label or "LoRA" in label else None)
            if j < 5:
                ax.annotate("", xy=(5, y-0.9), xytext=(5, y-0.2),
                           arrowprops=dict(arrowstyle="->", color="#444"))

    elif i == 1:  # Loss
        tokens = ["<im_start>","You","[CANBK","EMA:","action","HOLD","reasoning"]
        losses = [0, 0, 0, 0, 1.2, 0.3, 0.8]
        colors_bar = ["#334155" if l==0 else "#ef5350" for l in losses]
        ax.bar([j+1 for j in range(len(tokens))], [max(l,0.1) for l in losses],
               color=colors_bar, alpha=0.8, width=0.7)
        for j, (t, l) in enumerate(zip(tokens, losses)):
            ax.text(j+1, -0.5, t, ha="center", va="top", color="white", fontsize=7, rotation=45)
            if l > 0:
                ax.text(j+1, l+0.05, f"{l:.1f}", ha="center", va="bottom", color="white", fontsize=8)
        ax.axhline(sum(losses)/len(losses), color="#f59e0b", ls="--", lw=1.5, label="mean loss")
        ax.set_ylim(-1.5, 2); ax.tick_params(colors="white")
        ax.set_yticks([0, 0.5, 1.0, 1.5]); ax.yaxis.set_tick_params(labelcolor="white")
        ax.text(5, 9, "Only ?-entropy on\nassistant tokens", ha="center",
                color="#ef5350", fontsize=9)

    elif i == 2:  # Backward
        # Show gradient flow through LoRA
        for y, label, has_grad in [
            (8.5, "Loss (scalar)", True),
            (7.0, "LM Head grad", True),
            (5.5, "Layer 32..1 grad", True),
            (4.0, "B grad (?L/?B)", True),
            (3.0, "A grad (?L/?A)", True),
            (1.5, "W -- FROZEN (no grad)", False),
        ]:
            c = "#26a69a" if has_grad else "#444"
            ax.text(5, y, label, ha="center", va="center", color=c, fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="#1e2230",
                             edgecolor=c, alpha=0.8))
        ax.text(5, 0.5, "Only A,B accumulate\n?L/?? for optimizer", ha="center",
                color="#f59e0b", fontsize=9)

    elif i == 3:  # Update
        x = np.linspace(-3, 3, 100)
        # Simulate parameter converging
        for ep, c_line in [(0, "#ef535055"), (3, "#f59e0b55"), (7, "#26a69a")]:
            noise = 0.3 * np.exp(-ep*0.3)
            y_curve = np.exp(-((x-1.5)**2)/(2*(0.5+noise)**2))
            ax.plot(x*1.5+5, y_curve*6+1, color=c_line, lw=1.5)
        ax.text(7.3, 7.5, "converged", color="#26a69a", fontsize=8)
        ax.text(4, 7.5, "init", color="#ef5350", fontsize=8)
        ax.text(5, 0.5, "Adam lr=2e-4\nadamw_bnb_8bit (saves 50% vs fp32)", 
                ha="center", color="#94a3b8", fontsize=8)
        ax.text(5, 9.2, "Loss surface (simplified)", ha="center", color="white", fontsize=9)

for ax in axes:
    for sp in ax.spines.values(): sp.set_edgecolor("#2a2e39")

fig.suptitle("One training step: Forward -> Loss -> Backward -> Update (repeats until convergence)",
             color="white", fontsize=12)
plt.tight_layout()
plt.savefig("training_loop.png", dpi=120, bbox_inches="tight", facecolor="#0f1117")
plt.show()

---
## Chapter 8 -- MLflow: Tracking Every Experiment

**Library:** `mlflow`

### Why track experiments?

After Round 1 training you get val_loss = 0.51.
After Round 2 training you get val_loss = 0.47.
After changing lr from 2e-4 to 1e-4 you get val_loss = 0.49.

Without tracking, you forget which hyperparameters produced which results.
MLflow answers: "which run was best, what parameters did it use?"

### What we track

```python
mlflow.log_params({
    "method":         "qlora_4bit",       <- which training path
    "model":          "Nemotron-Nano-4B",
    "lora_r":         16,
    "epochs":         3,
    "learning_rate":  2e-4,
    "n_train":        487,               <- how many training examples
    "hardware":       "M5 Air 16GB",
    "token_strategy": "compact_prompt",  <- 188 vs 990 tokens
})

mlflow.log_metrics({           <- logged every N steps during training
    "train_loss":    0.42,
    "val_loss":      0.51,
    "json_validity": 0.97,     <- does model always output valid JSON?
    "schema_accuracy": 0.93,   <- are all required fields present?
})
```

### The MLflow model registry

After each round, register the adapter as a versioned model:
```
nemotron-trading
  +-- Version 1  (Round 1, val_loss=0.51)
  +-- Version 2  (Round 2, val_loss=0.47)  <- "Staging"
  +-- Version 3  (Round 3, val_loss=0.44)  <- "Production"
```

`mlflow ui` at http://localhost:5000 shows everything in a web dashboard.

In [ ]:
# Demonstrate MLflow tracking (works even without GPU -- logs synthetic metrics)
import json
from pathlib import Path
from datetime import datetime

try:
    import mlflow
    
    mlflow.set_experiment("trading-agent-finetune-demo")
    
    # Simulate 3 training rounds
    rounds_data = [
        {"round": 1, "lora_r": 16, "lr": 2e-4, "n_train": 87,  "val_loss": 0.68, "json_ok": 0.72},
        {"round": 2, "lora_r": 16, "lr": 2e-4, "n_train": 187, "val_loss": 0.54, "json_ok": 0.85},
        {"round": 3, "lora_r": 16, "lr": 2e-4, "n_train": 387, "val_loss": 0.44, "json_ok": 0.95},
    ]
    
    run_ids = []
    for r in rounds_data:
        with mlflow.start_run(run_name=f"round-{r['round']}") as run:
            # Log hyperparameters (once per run)
            mlflow.log_params({
                "round":     r["round"],
                "lora_r":    r["lora_r"],
                "lr":        r["lr"],
                "n_train":   r["n_train"],
                "method":    "qlora_4bit",
                "model":     "Nemotron-Nano-4B-Instruct",
            })
            # Simulate logging metrics per step
            for step in range(0, 200, 20):
                decay = np.exp(-step / 100)
                mlflow.log_metrics({
                    "train_loss":     r["val_loss"] + 0.15 * decay + np.random.randn() * 0.02,
                    "val_loss":       r["val_loss"] + 0.05 * decay + np.random.randn() * 0.01,
                    "json_validity":  min(1.0, r["json_ok"] - 0.1*decay + np.random.randn()*0.01),
                }, step=step)
            run_ids.append(run.info.run_id)
    
    print(f"Logged {len(run_ids)} training runs to MLflow.")
    print(f"View at: mlflow ui  ->  http://localhost:5000")
    print()
    print("Comparing runs:")
    print(f"{'Round':<8} {'val_loss':<12} {'json_ok':<10} {'n_train':<10}")
    print("-" * 45)
    for r in rounds_data:
        print(f"  {r['round']:<6} {r['val_loss']:<12.2f} {r['json_ok']:<10.2f} {r['n_train']:<10}")
    print()
    print("Key insight: more training data -> lower val_loss + better JSON validity")

except ImportError:
    print("pip install mlflow to enable experiment tracking")
    print("Then run: mlflow ui")

---
## Chapter 9 -- Saving and Deploying the Adapter

### What gets saved

After training, PEFT saves two files:
```
output/nemotron-trading/final_adapter/
+-- adapter_config.json      <- LoRA configuration (r, alpha, target_modules...)
+-- adapter_model.safetensors <- The trained A and B matrices (~60 MB)
```

The base model (8 GB) is **not saved** -- it's downloaded from HuggingFace each time.
You only need to distribute the 60 MB adapter.

### Three deployment options

**Option A: Adapter-only (smallest, slowest at inference)**
```python
from peft import PeftModel
model = AutoModelForCausalLM.from_pretrained("nvidia/Nemotron-Nano-4B-Instruct")
model = PeftModel.from_pretrained(model, "output/nemotron-trading/final_adapter")
```

**Option B: Merged model (one file, faster inference)**
```python
merged = model.merge_and_unload()   # fuses A and B into W: new W = W + BxA
merged.save_pretrained("output/nemotron-trading/merged")
```
After merging: W_new = W_original + BxA (stored as a single matrix, no overhead at inference)

**Option C: vLLM server (production, highest throughput)**
```bash
vllm serve output/nemotron-trading/merged \
    --port 8001 \
    --served-model-name nemotron-trading \
    --max-model-len 2048

# In model_config.json:
# "company": {"provider": "vllm", "model": "nemotron-trading"}
```

### How merging works mathematically

Before merge (LoRA): `output = Wxx + (BxA)xx`
After merge:         `output = (W + BxA)xx`

The new weight matrix `W_new = W + BxA` is computed once and stored.
At inference, there's no extra A/B computation -- it's just one matrix multiply.
This is why merged models are faster than adapter-only models.

In [ ]:
# Demonstrate adapter saving and merging (no GPU needed -- pure numpy)
import numpy as np
from pathlib import Path

print("Simulating LoRA adapter save/load/merge...")
print()

# Simulate a small attention layer (d=128 instead of 4096 for demo)
d, r = 128, 8
np.random.seed(42)

# Original frozen weight matrix
W_original = np.random.randn(d, d) * 0.02  # typical LLM init scale

# After training: LoRA A and B have been optimised
A_trained = np.random.randn(r, d) * 0.01   # rank r x d_in
B_trained = np.zeros((d, r))               # starts at 0, gets updated
# Simulate some learning happening
B_trained += np.random.randn(d, r) * 0.005
alpha = 16.0
scale = alpha / r                           # = 16/8 = 2.0

# -- Adapter-only inference --------------------------------------------------
x = np.random.randn(d)
out_base    = W_original @ x
out_adapted = W_original @ x + (B_trained @ A_trained @ x) * scale

print(f"Adapter-only inference:")
print(f"  Base output norm:    {np.linalg.norm(out_base):.4f}")
print(f"  Adapted output norm: {np.linalg.norm(out_adapted):.4f}")
print(f"  Adapter contribution: {np.linalg.norm(out_adapted - out_base):.4f}")
print(f"  Relative change: {np.linalg.norm(out_adapted-out_base)/np.linalg.norm(out_base)*100:.2f}%")
print()

# -- Merge: W_new = W_original + BxAxscale -----------------------------------
W_merged = W_original + (B_trained @ A_trained) * scale
out_merged = W_merged @ x

# Should be identical to adapter-only
diff = np.max(np.abs(out_adapted - out_merged))
print(f"Merged model inference:")
print(f"  Merged output norm:  {np.linalg.norm(out_merged):.4f}")
print(f"  Matches adapter-only: {diff < 1e-10} (diff={diff:.2e})")
print()

# -- File sizes --------------------------------------------------------------
bytes_W_bf16    = d * d * 2          # original base layer
bytes_A_bf16    = r * d * 2          # adapter A
bytes_B_bf16    = d * r * 2          # adapter B
bytes_W_merged  = d * d * 2          # merged = same as original

print(f"Memory comparison for one attention layer (d={d}, r={r}):")
print(f"  Original W:          {bytes_W_bf16:>8,} bytes")
print(f"  Adapter (A+B):       {bytes_A_bf16+bytes_B_bf16:>8,} bytes ({(bytes_A_bf16+bytes_B_bf16)/bytes_W_bf16*100:.1f}% of W)")
print(f"  Merged W (W + BxA):  {bytes_W_merged:>8,} bytes")
print()
print(f"For full Nemotron-4B (7 layers x 32 blocks):")
real_d, real_r = 4096, 16
n_lora_layers = 7 * 32   # 7 projections per block x 32 blocks
adapter_total = 2 * real_r * real_d * 2 * n_lora_layers  # A + B per layer
model_total   = 4e9 * 2   # ~4B params in bfloat16
print(f"  Full model:  {model_total/1e9:.1f} GB")
print(f"  Adapter:     {adapter_total/1e6:.0f} MB ({adapter_total/model_total*100:.2f}% of model)")
print()
print("This is why we distribute 60 MB adapters, not 8 GB full models.")

---
## Chapter 10 -- Run Training: The Complete Flow

This chapter shows exactly what happens when you run `python train.py`,
walking through every function call from argument parsing to the saved adapter.

### The decision tree inside train.py

```
python train.py --train data/trading_train.jsonl
        |
        v
detect_hardware()
        |
  Apple Silicon?  -----> train_mlx_lora()
  (arm64 + Darwin)         mlx_lm.lora subprocess
        |
  NVIDIA CUDA?
        |
    VRAM >= 16 GB? --> train_hf_lora()
                        bfloat16 frozen base + LoRA
        |
    VRAM 8-16 GB?  --> train_hf_qlora(bits=8)
                        int8 frozen + LoRA
        |
    VRAM < 8 GB?   --> train_hf_qlora(bits=4)
                        NF4 frozen + LoRA (QLoRA)
        |
  CPU only?        --> train_hf_lora() (slow)
```

### What happens inside train_hf_qlora() step by step

**Step 1: Load tokenizer**
```python
tokenizer = AutoTokenizer.from_pretrained("nvidia/Nemotron-Nano-4B-Instruct")
# Adds pad_token = eos_token (Nemotron has no separate pad token)
```

**Step 2: Load model in 4-bit NF4**
```python
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal Float 4
    bnb_4bit_compute_dtype=torch.bfloat16,  # dequantise to bf16 for compute
    bnb_4bit_use_double_quant=True,     # also quantise the quant constants
)
model = AutoModelForCausalLM.from_pretrained(
    "nvidia/Nemotron-Nano-4B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",  # spreads across available GPUs automatically
)
# Result: model is ~2.5 GB vs 8 GB in bfloat16
```

**Step 3: Prepare for k-bit training**
```python
model = prepare_model_for_kbit_training(model)
# Casts all LayerNorm to float32 (needed for stable gradients)
# Enables gradient checkpointing
# Marks the output embedding as requiring gradient
```

**Step 4: Attach LoRA adapters**
```python
lora_config = LoraConfig(
    r=16,           # rank
    lora_alpha=32,  # 2x rank
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Output: trainable params: 40,042,496 || all params: 4,083,036,160 || trainable: 0.98%
```

**Step 5: Load and format dataset**
```python
train_ds = load_dataset("json", data_files="data/trading_train.jsonl", split="train")
# Apply chat template: {"messages":[...]} -> {"text": "<|im_start|>system..."}
train_ds = train_ds.map(lambda ex: {
    "text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)
})
```

**Step 6: Configure SFTTrainer**
```python
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,    # each example is independent
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,  # effective batch = 16
        learning_rate=2e-4,
        num_train_epochs=3,
        bf16=True,
        optim="adamw_bnb_8bit",   # 8-bit Adam saves optimizer memory
        gradient_checkpointing=True,
        logging_steps=10,
        save_steps=100,
    )
)
```

**Step 7: Train**
```python
trainer.train()
# For each batch:
#   forward pass: W*x + (B*A)*x*scale  (per LoRA layer)
#   loss: cross-entropy on assistant tokens only
#   backward: compute gradients for A and B only
#   step: update A and B via 8-bit Adam
```

**Step 8: Save adapter**
```python
model.save_pretrained("output/nemotron-trading/final_adapter")
tokenizer.save_pretrained("output/nemotron-trading/final_adapter")
# Saves: adapter_config.json + adapter_model.safetensors (~60 MB)
```

### What Axolotl does differently

Axolotl wraps all of the above in a YAML config + one command.
It adds production features like flash attention, packing (multiple short sequences per batch),
gradient noise scheduling, and better checkpoint management. The actual PyTorch operations
are identical -- Axolotl just handles the engineering boilerplate.

### Estimated training times

| Hardware | Method | 500 examples, 3 epochs |
|---|---|---|
| M5 Air 16GB | MLX LoRA (compact prompts) | 1.5--2 hours |
| M5 Air 16GB | MLX LoRA (verbose prompts) | 6--8 hours |
| Colab T4 8GB | QLoRA 4-bit | 2.5--3 hours |
| Colab A10G 24GB | QLoRA 8-bit | 1.5 hours |
| Colab A100 40GB | LoRA bfloat16 | 45 minutes |

In [ ]:
import sys, os
sys.path.insert(0, "..")

try:
    from train import detect_hardware
    hw = detect_hardware()
    print("Hardware detected:")
    for k, v in hw.items():
        print(f"  {k}: {v}")
    method = hw["recommended_method"]
    print()
    print(f"Recommended method: {method}")
    if method == "mlx_lora":
        print("Command: python train.py --train data/trading_train.jsonl")
        print("         (MLX LoRA -- native Apple Silicon, no CUDA)")
    elif method == "qlora_4bit":
        print("Command: python train.py --train data/trading_train.jsonl")
        print("         (QLoRA 4-bit NF4 -- NVIDIA GPU with limited VRAM)")
    elif method == "lora":
        print("Command: python train.py --train data/trading_train.jsonl")
        print("         (LoRA bfloat16 -- NVIDIA GPU >=16GB VRAM)")
except Exception as e:
    print(f"Run from llm_finetune/ directory: {e}")

In [ ]:
# Training command quick reference
cmds = {
    "Apple M5 Air (auto)":
        "python train.py --train data/trading_train.jsonl --val data/trading_val.jsonl",
    "Force MLX (Mac)":
        "python train.py --train data/trading_train.jsonl --force_method mlx_lora",
    "Force QLoRA 4-bit (Colab T4)":
        "python train.py --train data/trading_train.jsonl --force_method qlora_4bit",
    "Force LoRA (Colab A100)":
        "python train.py --train data/trading_train.jsonl --force_method lora",
    "Axolotl (one YAML command)":
        "axolotl train axolotl_config.yaml",
    "With MLflow tracking":
        "MLFLOW_TRACKING_URI=http://localhost:5000 python train.py --train data/trading_train.jsonl",
}
print("Training command reference:")
print("=" * 60)
for label, cmd in cmds.items():
    print(f"
{label}:
  {cmd}")

---
## Chapter 11 -- Summary and Next Steps

### The complete production stack

```
WHAT                     LIBRARY              WHY
-----------------------------------------------------------------
Simulation data          yfinance             Historical OHLCV + fundamentals
Fast DataFrames          Polars               5-10x faster than pandas
Candle queries           DuckDB               SQL on Parquet, no full load
Experiment tracking      MLflow               Params, metrics, model registry
Fine-tuning config       Axolotl              One YAML = full training pipeline
Fine-tuning (Mac)        mlx-lm               Native Apple Silicon LoRA
Fine-tuning (NVIDIA)     peft + trl + bnb     QLoRA with HuggingFace stack
LoRA adapters            peft                 Wrap attention layers with A*B
Quantization             bitsandbytes         4-bit NF4, CUDA only
Production serving       vLLM                 10-50x faster than Ollama
Pipeline scheduling      Prefect              Daily scoring, weekly fine-tuning
Data versioning          DVC                  Reproducible experiments
Drift monitoring         Evidently AI         Know when to retrain
```

### The flywheel outcome (expected metrics)

After 5 rounds (250 episodes per symbol, 10 symbols = 2500 total):

| Round | val_loss | json_validity | win_rate |
|---|---|---|---|
| Baseline (fallback) | -- | 0% (no LLM) | ~45% |
| Round 1 | ~0.68 | ~72% | ~48% |
| Round 2 | ~0.54 | ~85% | ~52% |
| Round 3 | ~0.44 | ~95% | ~55% |
| Round 5 | ~0.35 | ~98% | ~58% |

### Interview cheat sheet

| Question | Key answer points |
|---|---|
| What is LoRA? | Freeze W, train A*B where A=(r x d_in), B=(d_out x r), r << d. 0.78% params per layer. |
| What is QLoRA? | LoRA + compress frozen W to 4-bit NF4. 8x memory saving. ~2% quality cost. |
| Which transformer parts? | Q,K,V,O projections + gate/up/down FFN. 7 layers x 32 blocks = 224 LoRA pairs. |
| Memory breakdown? | Model + Optimizer (Adam=2x params) + Gradients + Activations. QLoRA: ~5.5 GB total. |
| How does Adam work? | First moment (gradient EMA) + second moment (squared gradient EMA). Adaptive per-param LR. |
| What is grad checkpointing? | Don't store activations; recompute on backward. 50% less memory, 15% slower. |
| Why only assistant tokens? | Loss mask. Teach to decide, not memorise inputs. Better gradient density. |
| How does merging work? | W_new = W + B*A. Computed once. Single matrix at inference. Zero overhead. |
| How do you track experiments? | MLflow. Params, metrics, artifacts logged per run. Model registry for versioning. |
| When do you retrain? | Evidently AI drift detection. Action distribution shift = regime change = retrain. |

### Next steps

1. Run the simulation: `FALLBACK_MODE=True` in `trading_agent_sim.ipynb`
2. Build dataset: `python data_pipeline.py --db trading_agent.db --out data/training.jsonl`
3. Train: `python train.py --train data/training_train.jsonl --val data/training_val.jsonl`
4. Start MLflow: `mlflow ui` (http://localhost:5000)
5. Evaluate: `python evaluate.py --adapter output/nemotron-trading/final_adapter --test data/training_test.jsonl`
6. Deploy: set `provider: vllm` in `model_config.json`, run `vllm serve output/.../merged`
7. Monitor: `python monitoring.py --reference data/training_train.jsonl --current data/new_decisions.jsonl`
8. Repeat the flywheel